# COS30049 A2 - Training stages C1-C6 of the DS1 scoring methods, with their dev gate (Kaggle CPU session)

Serves `docs/TRAINING_PLAN.md` (sections 3-4). From the repository at a fixed commit, this rebuilds the DS1 features
and the tuning plants, then `stage-search` searches each scoring method's grid (`config/settings.json ->
stageSearch`) at the method's own M0 flag budget: candidates are ranked on tuning seeds 40000-40002, and the winner is
gated against M0 on tuning seeds 40003-40004 (`explorer/stage_search.py`). The reporting seeds are never read. Each
stage runs as its own step, so one failing stage does not stop the others. CPU only, so no GPU quota.

**Settings:** Accelerator **None**, **Internet on**. Then **Save Version -> Save & Run All (Commit)**. About an hour.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "d1fd31ef9a75617e37e4a66ba8e40f74063ecaf9"   # the code this run uses
STAGES = ["C1", "C2", "C3", "C4", "C5", "C6"]
STEPS = ["features", "split", "inject"]   # the data the search reads (M0's dev tables are committed)
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2"]

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

In [ ]:
def cli(*args, log):
    sh(sys.executable, "-m", "explorer.cli", *args, log=log)


for step in STEPS:
    cli(*step.split(), log=f"{step}.log")
failed = {}
for stage in STAGES:
    t = time.time()
    try:
        cli("stage-search", "--stages", stage, log=f"stage-search-{stage}.log")
    except subprocess.CalledProcessError as e:
        failed[stage] = str(e)
    print(f"{stage}: {(time.time() - t) / 60:.1f} min{' FAILED' if stage in failed else ''}")

In [ ]:
# The searches, the winners, the dev tables and stages.csv (to commit), the settings and the logs.
for rel in ["reports/tables/stage_search", "reports/tables/stage_dev", "reports/tables/stages.csv", "models/stage",
            "reports/logs", "config/settings.json"]:
    src = SRC / rel
    dest = WORK / "stage" / rel
    if src.is_dir():
        shutil.copytree(src, dest, dirs_exist_ok=True)
    elif src.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest)
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "stages": STAGES, "failed": failed,
    "steps": STEPS, "hours": round((time.time() - T0) / 3600, 2),
    "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS}}, indent=2), encoding="utf-8")
print((SRC / "reports" / "tables" / "stages.csv").read_text(encoding="utf-8"))